# Assignment 5: Predicting Class Label

**Sally Choi**  

**DATASCI 530**

## Introduction

This report explores whether a flower's species can be predicted from its sepal and petal measurements using the Iris dataset from the UCI Machine Learning Repository. Each observation represents one flower, with four measurements recorded in centimeters and a class label identifying the species as setosa, versicolor, or virginica.

### Key findings so far

The dataset contains **150 flowers**, with **50 observations in each species**, so all three classes are equally represented. All four predictors are numeric, and neither the measurements nor the class labels contain missing values. The descriptive analysis shows that setosa has much smaller petals on average than the other two species, while virginica has the largest average petal measurements. These differences suggest that petal measurements may help distinguish species, although model performance will need to be evaluated through cross-validation.


## Dataset description

### Load the data

The [UCI Iris dataset](https://archive.ics.uci.edu/dataset/53/iris) provides four predictors: sepal length, sepal width, petal length, and petal width. We store these measurements in `X` and the species labels in `y`, keeping the outcome separate from the information used to predict it. The class label is therefore not included in the predictor count.


In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from IPython.display import display

# Load the Iris dataset from UCI.
iris = fetch_ucirepo(id=53)
X = iris.data.features
y = iris.data.targets

### Dataset size and initial checks

Before describing the flowers, we check that every row of measurements has a matching class label and look for missing values in both parts of the dataset. These checks help establish whether any observations need to be removed or values filled in before analysis. Table 1 summarizes the dataset's size and completeness.


In [2]:
# Check that each flower has a matching class label.
assert X.index.equals(y.index), "Predictors and class labels must have matching rows."

# Summarize the size of the dataset and check for missing values.
dataset_summary = pd.DataFrame({
    "Measure": ["Observations (rows)", "Predictor features", "Missing predictor values", "Missing class labels"],
    "Count": [X.shape[0], X.shape[1], X.isna().sum().sum(), y.isna().sum().sum()]
})

display(dataset_summary.style.hide(axis="index").set_caption("Table 1. Dataset size and missing-value checks"))


Measure,Count
Observations (rows),150
Predictor features,4
Missing predictor values,0
Missing class labels,0


### Class distribution

Table 2 shows how many flowers belong to each species and what share of the sample they represent. We calculate each proportion by dividing the species count by the total number of flowers and display it as a percentage.


In [3]:
# Count the flowers in each class.
class_counts = y["class"].value_counts().sort_index()

# Divide each count by the total number of flowers to find its proportion.
class_summary = pd.DataFrame({
    "Class": class_counts.index.str.replace("Iris-", "", regex=False).str.capitalize(),
    "Flowers": class_counts.to_numpy(),
    "Proportion": (class_counts / len(y)).to_numpy()
})

display(class_summary.style.format({"Proportion": "{:.1%}"}).hide(axis="index")
        .set_caption("Table 2. Number and proportion of flowers in each class"))


Class,Flowers,Proportion
Setosa,50,33.3%
Versicolor,50,33.3%
Virginica,50,33.3%


The dataset has **150 rows and 4 predictor features**, with an additional column containing the target class. Each species contributes **50 flowers, or 33.3% of the sample**, so there is no majority class. For context, a classifier that always predicts the same species would correctly classify only one-third of these observations.

The predictor and target indices match, and there are no missing measurements or labels. As a result, no imputation is needed, and all observations are available for the descriptive analysis.


### Predictor types, ranges, and variation

To understand what the models will use, we examine each predictor's data type and summarize its distribution. Table 3 reports the mean, standard deviation, median, minimum, and maximum for each measurement. The standard deviation describes how much measurements vary around their mean, while the minimum and maximum help us check the observed range. We also count zero or negative measurements, since flower lengths and widths should be positive.


In [4]:
# Summarize the four numeric predictors; all measurements are in centimeters.
predictor_summary = X.agg(["mean", "std", "median", "min", "max"]).T
predictor_summary.columns = ["Mean", "Std. deviation", "Median", "Minimum", "Maximum"]
predictor_summary.insert(0, "Data type", X.dtypes.astype(str))
predictor_summary.index = predictor_summary.index.str.title()
predictor_summary.index.name = "Predictor (cm)"

display(predictor_summary.style.format(precision=2)
        .set_caption("Table 3. Predictor types and descriptive statistics"))

# Flag measurements that would be inconsistent with a positive length or width.
print(f"Zero or negative measurements: {(X <= 0).sum().sum()}")


,Data type,Mean,Std. deviation,Median,Minimum,Maximum
Predictor (cm),,,,,,
Sepal Length,float64,5.84,0.83,5.80,4.30,7.90
Sepal Width,float64,3.05,0.43,3.00,2.00,4.40
Petal Length,float64,3.76,1.76,4.35,1.00,6.90
Petal Width,float64,1.20,0.76,1.30,0.10,2.50


Zero or negative measurements: 0


All four predictors are stored as numeric values, and none of the measurements are zero or negative. Sepal length ranges from **4.30 to 7.90 cm**, while petal length ranges from **1.00 to 6.90 cm**. Petal length also has the largest standard deviation (**1.76 cm**), indicating that its measurements vary more across the sample than those of the other predictors. These checks reveal no obvious nonpositive measurement errors, although positive values alone do not establish that every measurement is accurate.


### Repeated observations

We also check for rows with identical measurements and the same species label. Because the dataset does not provide an individual flower identifier, identical records could represent different flowers with the same recorded measurements. We therefore report these repeats and retain them rather than assuming that they are data-entry errors.


In [5]:
# Combine predictors and labels to check complete records and compare species.
iris_data = pd.concat([X, y], axis=1)

# Count extra copies after the first occurrence of each identical record.
repeated_rows = iris_data.duplicated().sum()
print(f"Repeated rows beyond the first occurrence: {repeated_rows}")


Repeated rows beyond the first occurrence: 3


There are **3 repeated rows beyond their first occurrence** when all four measurements and the class label are considered together. We retain all 150 observations because matching recorded measurements do not establish that the same flower was entered twice. However, identical records could appear in different cross-validation folds, so these repeats should be kept in mind when interpreting later model results.


### Average predictor measurements by species

The overall averages combine three species that may have different physical characteristics. Table 4 separates them by calculating the mean of each predictor within each class, making it easier to see which measurements might help distinguish the flowers. Values are rounded to two decimal places and remain in centimeters.


In [6]:
# Calculate the average of each predictor separately for each species.
class_means = iris_data.groupby("class")[X.columns].mean()

# Use readable species names and column labels for the report.
class_means.index = class_means.index.str.replace("Iris-", "", regex=False).str.capitalize()
class_means.index.name = "Species"
class_means.columns = class_means.columns.str.title() + " (cm)"

display(class_means.style.format("{:.2f}")
        .set_caption("Table 4. Mean sepal and petal measurements by species"))


,Sepal Length (cm),Sepal Width (cm),Petal Length (cm),Petal Width (cm)
Species,,,,
Setosa,5.01,3.42,1.46,0.24
Versicolor,5.94,2.77,4.26,1.33
Virginica,6.59,2.97,5.55,2.03


Setosa stands out for its small petals: its mean petal length is **1.46 cm**, compared with **4.26 cm** for versicolor and **5.55 cm** for virginica. Petal width follows the same ordering, with means of **0.24 cm**, **1.33 cm**, and **2.03 cm**, respectively. Virginica also has the longest sepals on average (**6.59 cm**), while setosa has the widest (**3.42 cm**).

These differences suggest that petal measurements may be useful predictors of species. However, class averages do not show how much individual flowers overlap, so they cannot establish how accurately a model will classify a new flower. The later model comparison will test whether these descriptive patterns translate into predictive performance.
